# 04 — Model Comparison & Chemical Interpretation

ECFP-MLP vs GraphConv 비교, 레이더 차트, False Negative 분자, 작용기별 독성 해석.

⚠️ **노트북 02 와 03 을 먼저 실행**해 `*_preds.npz` 가 Drive 에 저장돼 있어야 합니다.

In [ ]:
# === STEP 1: 환경 설정 (가장 먼저 이 셀 실행) ===========================
import os, sys, subprocess

REPO = "Tox21-Toxicity-Classifier"
REPO_URL = "https://github.com/zqvo04/Tox21-Toxicity-Classifier.git"
ROOT = "/content/" + REPO

# (1) 레포 준비: 없으면 clone, 이미 있으면 최신으로 git pull(오래된 캐시 방지)
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", REPO_URL, ROOT], check=False)
else:
    subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("작업 디렉토리:", os.getcwd())

# (2) 의존성 설치 (idempotent). 'rdkit'(공식) 사용 — 'rdkit-pypi'는 최신
#     Python에서 설치 불가. 데이터/특징화는 RDKit으로 직접 처리(견고).
pkgs = ["rdkit", "torch-geometric",
        "pandas", "scikit-learn", "matplotlib", "seaborn", "tqdm"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# (3) 코드 버전 검증: 최신 RDKit 기반 dataset.py 인지 확인 (deepchem 의존 없음)
import importlib
import src.dataset as _ds
importlib.reload(_ds)
if not hasattr(_ds, "load_tox21_dataframe"):
    raise RuntimeError(
        "⚠️ 오래된 dataset.py 입니다 (deepchem 버전). 다음을 확인하세요:\n"
        "  1) 최신 코드를 GitHub 에 push 했는지\n"
        "  2) Colab 상단 [런타임 > 세션 다시 시작 및 모두 삭제] 후 이 셀을 다시 실행\n"
        "     (오래된 clone 폴더가 남아 있으면 갱신이 안 됩니다)")

# (4) import 점검. 실패 시 런타임 재시작 안내.
try:
    import rdkit, torch, torch_geometric, sklearn
    print("✅ rdkit", rdkit.__version__,
          "| torch", torch.__version__, "| pyg", torch_geometric.__version__,
          "| gpu", torch.cuda.is_available(), "| dataset.py: 최신(RDKit) ✔")
except Exception as e:
    print("⚠️ import 실패:", repr(e))
    print("➡️  상단 메뉴 [런타임 > 런타임 다시 시작] 후, 이 셀을 한 번 더 실행하세요.")

In [ ]:
# === STEP 2: Google Drive 마운트 (체크포인트/결과 저장) ==================
# 경로를 본인 Drive 폴더로 바꿔도 됩니다. 02·03·04 에서 동일하게 유지하세요.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/tox21/'
except Exception as e:
    # Colab이 아닌 환경이면 로컬 폴더로 대체
    print("Drive 마운트 생략 (로컬 폴더 사용):", repr(e))
    CKPT_DIR = './checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)
print("CKPT_DIR =", CKPT_DIR)

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid')
from src import evaluate
FIG = 'results/figures'; os.makedirs(FIG, exist_ok=True)

## 1. 저장된 예측 로딩

In [ ]:
def load_preds(prefix):
    path = os.path.join(CKPT_DIR, f'{prefix}_preds.npz')
    assert os.path.exists(path), f'{path} 없음 → 노트북 02/03 을 먼저 실행하세요.'
    d = np.load(path, allow_pickle=True)
    return d['probs'], d['y_true'], d['mask'], list(d['tasks'])

ecfp_probs, y_true, mask, tasks = load_preds('ecfp_mlp')
gcn_probs, y_true_g, mask_g, _   = load_preds('graphconv')

df_ecfp, sum_ecfp = evaluate.evaluate(y_true,   ecfp_probs, mask,   tasks, name='ECFP-MLP')
df_gcn,  sum_gcn  = evaluate.evaluate(y_true_g, gcn_probs,  mask_g, tasks, name='GraphConv')
print(sum_ecfp); print(sum_gcn)

## 2. 성능 비교 테이블

In [ ]:
comp = pd.DataFrame({'task': tasks,
    'ECFP_ROC': df_ecfp['roc_auc'].values, 'GCN_ROC': df_gcn['roc_auc'].values,
    'ECFP_PR':  df_ecfp['pr_auc'].values,  'GCN_PR':  df_gcn['pr_auc'].values})
comp.loc['MEAN'] = ['MEAN', np.nanmean(comp['ECFP_ROC']), np.nanmean(comp['GCN_ROC']),
                    np.nanmean(comp['ECFP_PR']), np.nanmean(comp['GCN_PR'])]
comp.to_csv(f'{FIG}/comparison_table.csv', index=False)
display(comp.round(4))

## 3. 레이더 차트 (per-task ROC-AUC)

In [ ]:
labels = tasks
e = np.nan_to_num(df_ecfp['roc_auc'].values, nan=0.5)
g = np.nan_to_num(df_gcn['roc_auc'].values, nan=0.5)
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
e = np.concatenate([e, [e[0]]]); g = np.concatenate([g, [g[0]]]); angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax.plot(angles, e, 'o-', lw=2, label='ECFP-MLP'); ax.fill(angles, e, alpha=0.15)
ax.plot(angles, g, 'o-', lw=2, label='GraphConv'); ax.fill(angles, g, alpha=0.15)
ax.set_thetagrids(np.degrees(angles[:-1]), labels)
ax.set_ylim(0.5, 1.0); ax.set_title('Per-task ROC-AUC', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1))
plt.tight_layout(); plt.savefig(f'{FIG}/radar_comparison.png', dpi=150); plt.show()

In [ ]:
x = np.arange(len(tasks)); w = 0.38
plt.figure(figsize=(14, 5))
plt.bar(x - w/2, np.nan_to_num(df_ecfp['roc_auc'],nan=0), w, label='ECFP-MLP')
plt.bar(x + w/2, np.nan_to_num(df_gcn['roc_auc'],nan=0),  w, label='GraphConv')
plt.xticks(x, tasks, rotation=45, ha='right'); plt.ylabel('ROC-AUC'); plt.ylim(0.5,1.0)
plt.legend(); plt.title('ROC-AUC by task'); plt.tight_layout()
plt.savefig(f'{FIG}/roc_auc_bar.png', dpi=150); plt.show()

## 4. False Negative 분자 분석
실제 독성인데 안전하다고 예측한(가장 위험한) 분자.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from src.dataset import get_test_smiles

# 테스트셋 SMILES (저장된 예측과 동일한 scaffold split 순서로 정렬됨)
test_ids = get_test_smiles()

task = 'SR-MMP'; ti = tasks.index(task)
fn_df = evaluate.find_false_negatives(y_true, ecfp_probs, mask, test_ids, ti, threshold=0.5)
print(f'False negatives on {task}:', len(fn_df)); display(fn_df.head(12))

mols, legs = [], []
for _, r in fn_df.head(8).iterrows():
    m = Chem.MolFromSmiles(r['smiles'])
    if m: mols.append(m); legs.append(f"p={r['prob']:.2f}")
img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(220,180), legends=legs)
try: img.save(f'{FIG}/false_negatives_{task}.png')
except AttributeError: open(f'{FIG}/false_negatives_{task}.png','wb').write(img.data)
img

## 5. 작용기별 독성 해석 (structural alert enrichment)
알려진 독성 작용기를 SMARTS 로 매칭해, 보유/비보유 분자의 독성률 비(enrichment)를 계산.

In [ ]:
from rdkit import Chem
alerts = {
    'Aromatic nitro':      '[$([NX3](=O)=O),$([NX3+](=O)[O-])][c]',
    'Aromatic amine':      'c[NX3;H2,H1]',
    'Michael acceptor':    '[CX3]=[CX3][CX3]=[OX1]',
    'Epoxide':             'C1OC1',
    'Halogenated aromatic':'c[F,Cl,Br,I]',
    'Phenol':              'c[OX2H]',
}
patt = {k: Chem.MolFromSmarts(v) for k, v in alerts.items()}
task = 'SR-MMP'; ti = tasks.index(task)
valid = mask[:, ti] > 0
smi_valid, lab_valid = np.asarray(test_ids)[valid], y_true[valid, ti]

rows = []
for name, p in patt.items():
    tox_has = tox_no = n_has = n_no = 0
    for smi, lab in zip(smi_valid, lab_valid):
        m = Chem.MolFromSmiles(smi)
        if m is None: continue
        if m.HasSubstructMatch(p): n_has += 1; tox_has += int(lab == 1)
        else:                      n_no  += 1; tox_no  += int(lab == 1)
    r_has = tox_has / max(n_has, 1); r_no = tox_no / max(n_no, 1)
    rows.append(dict(alert=name, n_with=n_has, tox_rate_with=round(r_has, 3),
                     tox_rate_without=round(r_no, 3),
                     enrichment=round(r_has / max(r_no, 1e-6), 2)))
alert_df = pd.DataFrame(rows).sort_values('enrichment', ascending=False)
alert_df.to_csv(f'{FIG}/structural_alerts.csv', index=False); display(alert_df)

### 화학적 해석 요약

- **Aromatic nitro / amine**: 대사 활성화로 반응성 중간체 형성 → 미토콘드리아 독성(SR-MMP)·유전독성과 연관.
- **Michael acceptor / epoxide**: 전자친화성으로 단백질·DNA 공유결합 → 스트레스 경로(SR-ARE, SR-p53) 활성화.
- **Halogenated aromatic**: 친유성↑ → 막 투과 및 AhR 수용체 결합과 관련.

**모델 관점**: ECFP-MLP 는 부분구조 alert 를 직접 비트로 인코딩해 해석성이 높고, GraphConv 는
메시지 전달로 더 유연한 표현을 학습하지만 해석성은 낮다. 작고 불균형한 Tox21 에서는 두 접근의
평균 ROC-AUC 가 비슷하게 수렴하는 경우가 많다.

## 6. 결론
비교 결과·그림은 `results/figures/` 에 저장되어 README 에서 참조된다.